# SalesLens - Data Cleaning

## 1. Cleaning Objectives


In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.clean_data import clean_datasets, export_cleaned, validate_cleaned_datasets, count_orphan_rows, DOCS_REPORT
import pandas as pd


## 2. Load Raw Data

In [ ]:
cleaned, stats = clean_datasets()
stats

## 3. Data Type Corrections

In [ ]:
for name in ['olist_orders_dataset.csv','olist_order_items_dataset.csv','olist_order_reviews_dataset.csv']:
    print(f'\n{name}')
    print(cleaned[name].dtypes)


## 4. Missing Values

In [ ]:
for name, df in cleaned.items():
    print(f'\n### {name}')
    print(df.isna().sum().sort_values(ascending=False).head(10))


## 5. Duplicate Records

In [ ]:
for name, df in cleaned.items():
    print(f'{name}: {df.duplicated().sum()} duplicates')


## 6. Invalid / Inconsistent Values

In [ ]:
checks = {
    'olist_order_items_dataset.csv': ['price', 'freight_value'],
    'olist_order_payments_dataset.csv': ['payment_installments', 'payment_value'],
    'olist_order_reviews_dataset.csv': ['review_score'],
    'olist_products_dataset.csv': ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
}
for name, cols in checks.items():
    df = cleaned[name]
    print(f'\n### {name}')
    for c in cols:
        print(c, 'min=', df[c].min(), 'max=', df[c].max())


## 7. Referential Integrity

In [ ]:
orders = cleaned['olist_orders_dataset.csv']
customers = cleaned['olist_customers_dataset.csv']
items = cleaned['olist_order_items_dataset.csv']
products = cleaned['olist_products_dataset.csv']
sellers = cleaned['olist_sellers_dataset.csv']
print('orders without customer:', count_orphan_rows(orders, customers[['customer_id']], 'customer_id'))
print('items without order:', count_orphan_rows(items, orders[['order_id']], 'order_id'))
print('items without product:', count_orphan_rows(items, products[['product_id']], 'product_id'))
print('items without seller:', count_orphan_rows(items, sellers[['seller_id']], 'seller_id'))


## 8. Cleaning Decisions

- Convert date columns to datetime.
- Remove exact geolocation duplicates.
- Keep missing review text fields and missing delivery timestamps.
- Preserve original product category and add English translation.

## 9. Export Processed Data

In [ ]:
export_cleaned(cleaned)
print('processed exported')

## 10. Validation

In [ ]:
validate_cleaned_datasets(cleaned)
print('validation passed')
print(DOCS_REPORT.read_text(encoding='utf-8').splitlines()[:10])
